# Cold-Start Airbnb Listing Predictor  

### Author: Andrea Suardi  
*Date: October - November 2025*  
[GitHub Repository](https://github.com/Andrea-Suardi/Airbnb-Project)  

---

## Project Overview  
This notebook explores how machine learning can address the **cold-start problem** in Airbnb listings — predicting whether newly added NYC properties are likely to be “High-Potential” (top 25% ratings, ≥4.9) before any reviews are available.  

By modeling listing characteristics such as **price, amenities, and location**, the goal is to help identify and promote trustworthy new properties, supporting early recommendations and improving user experience.

---

## Objectives  
- Build a binary classifier to distinguish *High-Potential* vs *Standard* listings.  
- Identify key predictive features influencing listing quality.  
- Evaluate model precision and interpretability through internal and prospective validation (on 2025 data).  
- Demonstrate an end-to-end workflow: preprocessing → feature engineering → modeling → evaluation.  

---

## Tools & Libraries  
- **Python:** Pandas, NumPy, Scikit-Learn, Matplotlib, Seaborn  
- **Environment:** Jupyter Notebook  
- **Additional:** Generative AI tools for ideation and documentation support  

---

## Notebook Structure  
1. **Data Understanding** 

---

## Notes  
This project is ongoing and currently focuses on data exploration and feature design.  
Subsequent updates will include model benchmarking and prospective validation.


In [60]:
# import required libraries
import pandas as pd
pd.set_option('display.max_rows', None)

# Data Understanding
In this section i will perform some checks to decide which data use for model development and for prospective evaluation

In [20]:
#import data
listings_2022 = pd.read_csv("NYC_2022.csv",low_memory=False)
listings_2023 = pd.read_csv("NYC_2024.csv",low_memory=False)
listings_2024 = pd.read_excel("NYC_2023.xlsx")
listings_2025 = pd.read_csv("NYC_2025.csv",low_memory=False)

In [24]:
print("2022 shape:",listings_2022.shape,"\n2023 shape:",listings_2023.shape,"\n2025 shape:",listings_2025.shape)

2022 shape: (37410, 74) 
2023 shape: (37548, 75) 
2025 shape: (36111, 79)


In [45]:
#find listings with not enough reviews for model development
few_review_2022 = listings_2022[listings_2022['number_of_reviews'] < 5]['id']
few_review_2023 = listings_2023[listings_2023['number_of_reviews'] < 5]['id']

print(f"2022: check\nPercentage of Listings with enough reviews: {len(few_review_2022)/len(listings_2022):.2%}, Listings with enough reviews: {len(listings_2022)-len(few_review_2022)}")
print(f"\n2023: check\nPercentage of Listings with enough reviews: {len(few_review_2023)/len(listings_2023):.2%}, Listings with enough reviews: {len(listings_2023)-len(few_review_2023)}")


2022: check
Percentage of Listings with enough reviews: 47.54%, Listings with enough reviews: 19625

2023: check
Percentage of Listings with enough reviews: 53.79%, Listings with enough reviews: 17351


In [82]:
# find persistent listings with enough revies in other year
persistents_22_23 = few_review_2022[few_review_2022.isin(listings_2023[listings_2023['number_of_reviews'] < 5]['id'])]
persistents_22_25 = few_review_2022[few_review_2022.isin(listings_2025[listings_2025['number_of_reviews'] < 5]['id'])]
persistents_23_25 = few_review_2023[few_review_2023.isin(listings_2025[listings_2025['number_of_reviews'] < 5]['id'])]

#compute Persistent listings Percentage

perc_pers_22_23 = len(persistents_22_23) / len(few_review_2022)
perc_pers_22_25 = len(persistents_22_25) / len(few_review_2022)
perc_pers_23_25 = len(persistents_23_25) / len(few_review_2023)

print(f"2022-2023:check\nPersistent listings Percentage: {perc_pers_22_23:.2%}, Persistents: {len(persistents_22_23)}")
print(f"\n2022-2025:check\nPersistent listings Percentage: {perc_pers_22_25:.2%}, Persistents: {len(persistents_22_25)}")
print(f"\n2023-2025:check\nPersistent listings Percentage: {perc_pers_23_25:.2%}, Persistents: {len(persistents_23_25)}")




2022-2023:check
Persistent listings Percentage: 48.22%, Persistents: 8576

2022-2025:check
Persistent listings Percentage: 44.41%, Persistents: 7898

2023-2025:check
Persistent listings Percentage: 75.09%, Persistents: 15166


In [79]:
# Check missing values in 2022 data
missing_percentage_series = (
    listings_2022[listings_2022['number_of_reviews'] >= 5]
    .isna()
    .sum() * 100 / len(listings_2022[listings_2022['number_of_reviews'] >= 5])
)

# Apply the sort_values method to sort the resulting Series
sorted_missing_percentage = missing_percentage_series[missing_percentage_series>5].sort_values(ascending=False)
print("2022 above 5% missings:\n",sorted_missing_percentage.map('{:.2f}%'.format))

2022 above 5% missings:
 calendar_updated         100.00%
bathrooms                100.00%
license                   99.98%
host_about                36.82%
neighborhood_overview     31.33%
neighbourhood             31.32%
host_response_time        24.79%
host_response_rate        24.79%
host_acceptance_rate      21.65%
host_neighbourhood        18.87%
bedrooms                   9.85%
dtype: object


In [80]:
# Check missing values in 2023 data
missing_percentage_series = (
    listings_2023[listings_2023['number_of_reviews'] >= 5]
    .isna()
    .sum() * 100 / len(listings_2023[listings_2023['number_of_reviews'] >= 5])
)

# Apply the sort_values method to sort the resulting Series
sorted_missing_percentage = missing_percentage_series[missing_percentage_series>5].sort_values(ascending=False)
print("2023 above 5% missings:\n",sorted_missing_percentage.map('{:.2f}%'.format))

2023 above 5% missings:
 calendar_updated         100.00%
license                   77.47%
host_about                39.95%
host_response_rate        34.86%
host_response_time        34.86%
neighborhood_overview     33.75%
neighbourhood             33.74%
host_acceptance_rate      33.57%
beds                      32.24%
bathrooms                 32.02%
price                     32.02%
host_neighbourhood        20.02%
host_location             18.26%
bedrooms                  12.49%
has_availability           6.18%
dtype: object


In [25]:
#check if the columns have same names
print("2022-2023:", set(listings_2022.columns) == set(listings_2023.columns))
print("2022-2024:", set(listings_2022.columns) == set(listings_2024.columns))
print("2022-2025:", set(listings_2022.columns) == set(listings_2025.columns))
print("2023-2024:", set(listings_2023.columns) == set(listings_2024.columns))
print("2023-2025:", set(listings_2023.columns) == set(listings_2025.columns))
print("2024-2025:", set(listings_2024.columns) == set(listings_2025.columns))

2022-2023: False
2022-2024: False
2022-2025: False
2023-2024: False
2023-2025: False
2024-2025: False


In [ ]:
print("columns from 2024 file that are not in others: ", set(listings_2024.columns) - set(listings_2023.columns))
print("\n number of missing columns: ", len(set(listings_2024.columns) - set(listings_2023.columns)))

There are problems with 2024 dataset, it is an excel file while others are .csv files

In [8]:
print("columns from 2024 file that are not in others: ", set(listings_2024.columns) - set(listings_2023.columns))
print("\n number of missing columns: ", len(set(listings_2024.columns) - set(listings_2023.columns)))

print("\n columns from other files that are not in 2024 file: ", set(listings_2022.columns) - set(listings_2024.columns))
print("\n number of missing columns: ",len(set(listings_2023.columns) - set(listings_2024.columns)))

columns from 2024 file that are not in others:  {'guests_included', 'market', 'jurisdiction_names', 'monthly_price', 'zipcode', 'notes', "SPACE: The apartment's furnishings are new and contemporary in design,with multi-channel CABLE television, DVD/video player and WiFi as well as wired internet all available for your entertainment. Local and domestic phone calls are also included.  This ap", 'square_feet', 'host_Total_listings_count', 'transit', 'access', 'state', 'summary', 'medium_url', 'thumbnail_url', 'features', 'street', 'geolocation', 'xl_picture_url', 'interaction', 'host_Since', 'experiences_offered', 'house_rules', 'cleaning_fee', 'neighbourhood_overview', 'weekly_price', 'security_deposit', 'country', 'bed_type', 'country_code', 'cancellation_policy', 'extra_people', 'city', 'smart_location'}

 number of missing columns:  34

 columns from other files that are not in 2024 file:  {'host_has_profile_pic', 'minimum_nights_avg_ntm', 'host_total_listings_count', 'maximum_maximum

there are lot of differences between 2024 data and others: 34 variables in 2024 that are not in others and 19 in others that are not in 2024. three of these differences are for different names.

In [85]:
#check columns names between 2022-2023-2025
print("columns from 2022 file that are not in 2023: ", set(listings_2022.columns) - set(listings_2023.columns))
print("columns from 2023 file that are not in 2022: ", set(listings_2023.columns) - set(listings_2022.columns))
print("columns from 2022 file that are not in 2025: ", set(listings_2022.columns) - set(listings_2025.columns))
print("columns from 2025 file that are not in 2022: ", set(listings_2025.columns) - set(listings_2022.columns))

columns from 2022 file that are not in 2023:  set()
columns from 2023 file that are not in 2022:  {'source'}
columns from 2022 file that are not in 2025:  set()
columns from 2025 file that are not in 2022:  {'number_of_reviews_ly', 'estimated_revenue_l365d', 'estimated_occupancy_l365d', 'source', 'availability_eoy'}


All columns in 2022 data are also in 2023 and 2025 data. There a few columns in 2023 and 2025 that are not in 2022 but this is irrilevant,

In [95]:
#find 75th percentile of review score in 2022 data
md_listings_2022=listings_2022[(listings_2022['number_of_reviews'] >= 5)].copy()
th_perc=md_listings_2022['review_scores_rating'].quantile(0.75)
print("75th percentile of observed distribution of review score: ",th_perc )

75th percentile of observed distribution of review score:  4.92


so the hybrid criterium "above 75th percentile and >4.9" reduces to just "above 75th percentile"

In [100]:
#check if target has imbalanced distribution in 2022 data
print(f"Percentage of High Potential Listings: {len(md_listings_2022[md_listings_2022['review_scores_rating']> th_perc])/len(md_listings_2022):.2%}")

Percentage of High Potential Listings: 22.60%


In [101]:
md_listings_2022.to_parquet('2022.parquet')
md_listings_2022 = pd.read_parquet('2022.parquet')

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.